# Day 30 — Global Health Data Exchange Data Day
### #30DayChartChallenge | April 2026

**The pandemic rewrote the leaderboard.**  Top global causes of disease burden (DALYs) in 2010, 2020, and 2021 — a three-column bump chart showing how COVID-19 ascended to #1, while diabetes and low back pain entered the top 10 and HIV/AIDS, malaria and diarrheal diseases dropped out.

**Tool:** `ggplot2` · 3-column bump chart with category-coloured connectors
**Data:** [IHME Global Burden of Disease Study 2021 — Findings Booklet](https://www.healthdata.org/sites/default/files/2024-05/GBD_2021_Booklet_FINAL_2024.05.16.pdf) (May 2024), p. 14. Original publication: GBD 2021 DALYs and HALE Collaborators, *The Lancet* 403:2133, 2024.
**Author:** Sharfudeen Yasar Arafath


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(showtext)
library(sysfonts)
library(scales)
library(ragg)

font_add_google("Outfit",          "outfit")
font_add_google("Roboto Condensed","roboto_condensed")
font_add_google("JetBrains Mono",  "jetbrains")
showtext_auto()
showtext_opts(dpi = 300)

# Dark theme — matches Days 1–27
bg_col   <- "#0a0e17"
text_col <- "#e0e6f0"
subtext  <- "#9eb3c8"
grid_col <- "#1a2030"

# Brighter category palette tuned for dark bg
cat_cols <- c(
  "CMNN"     = "#ff7e6b",   # Communicable / maternal / neonatal / nutritional
  "NCD"      = "#4ea1d9",   # Non-communicable disease
  "Injuries" = "#ffd166"    # Injuries
)

raw <- read.csv("../../data/day_30/gbd2021_dalys_ranks.csv",
                stringsAsFactors = FALSE)

years <- c(2010, 2020, 2021)
ranks <- raw %>%
  select(cause, category, rank_2010, rank_2020, rank_2021) %>%
  pivot_longer(starts_with("rank_"), names_to = "year",
               values_to = "rank", values_drop_na = TRUE) %>%
  mutate(year = as.integer(sub("rank_", "", year)))
dalys <- raw %>%
  select(cause, dalys_2010, dalys_2020, dalys_2021) %>%
  pivot_longer(starts_with("dalys_"), names_to = "year",
               values_to = "dalys", values_drop_na = TRUE) %>%
  mutate(year = as.integer(sub("dalys_", "", year)))
dat <- left_join(ranks, dalys, by = c("cause", "year"))
dat$y <- -dat$rank
dat$x <- match(dat$year, years)

movers <- dat %>% group_by(cause) %>%
  summarise(rmin = min(rank), rmax = max(rank)) %>%
  mutate(dramatic = (rmax - rmin) >= 5)
dat <- left_join(dat, movers %>% select(cause, dramatic), by = "cause")

left_pts  <- dat %>% filter(year == min(year)) %>%
  group_by(cause) %>% slice(1)
right_pts <- dat %>% filter(year == max(year)) %>%
  group_by(cause) %>% slice(1)

p <- ggplot(dat, aes(x = x, y = y, group = cause)) +
  geom_line(aes(colour = category, alpha = dramatic, linewidth = dramatic)) +
  geom_point(aes(colour = category, size = dramatic),
             fill = bg_col, shape = 21, stroke = 1.2) +
  geom_text(data = left_pts,
            aes(x = x - 0.06, y = y, label = cause, colour = category),
            family = "outfit", fontface = "bold", size = 3.4,
            hjust = 1, vjust = 0.5, show.legend = FALSE) +
  geom_text(data = right_pts,
            aes(x = x + 0.06, y = y,
                label = paste0(cause,
                               ifelse(rank <= 10, "",
                                      paste0("  (rank ", rank, ")"))),
                colour = category,
                fontface = ifelse(rank <= 10, "bold", "plain")),
            family = "outfit", size = 3.4, hjust = 0,
            vjust = 0.5, show.legend = FALSE) +
  annotate("text", x = 1, y = 2.0, label = "2010",
           family = "outfit", fontface = "bold",
           size = 6.5, colour = text_col) +
  annotate("text", x = 2, y = 2.0, label = "2020",
           family = "outfit", fontface = "bold",
           size = 6.5, colour = text_col) +
  annotate("text", x = 3, y = 2.0, label = "2021",
           family = "outfit", fontface = "bold",
           size = 6.5, colour = text_col) +
  annotate("text", x = 1, y = 1.0,
           label = "pre-pandemic baseline",
           family = "roboto_condensed", size = 3.0, colour = subtext,
           fontface = "italic") +
  annotate("text", x = 2, y = 1.0,
           label = "first pandemic year",
           family = "roboto_condensed", size = 3.0, colour = subtext,
           fontface = "italic") +
  annotate("text", x = 3, y = 1.0,
           label = "second pandemic year",
           family = "roboto_condensed", size = 3.0, colour = subtext,
           fontface = "italic") +
  geom_hline(yintercept = -10.5, colour = grid_col,
             linetype = "22", linewidth = 0.45) +
  annotate("text", x = 0.55, y = -10.85,
           label = "TOP 10 ↑    BELOW TOP 10 ↓",
           family = "jetbrains", size = 2.8, colour = subtext,
           hjust = 0, vjust = 1) +
  scale_x_continuous(limits = c(-0.6, 3.95),
                     expand = expansion(mult = c(0, 0))) +
  coord_cartesian(clip = "off") +
  scale_y_continuous(breaks = -seq(1, 24, 1),
                     labels = seq(1, 24, 1),
                     limits = c(-25, 3.2),
                     expand = expansion(mult = c(0.01, 0.01))) +
  scale_colour_manual(values = cat_cols,
                      labels = c(CMNN = "Communicable, maternal, neonatal, nutritional",
                                 NCD  = "Non-communicable disease",
                                 Injuries = "Injuries"),
                      name = NULL) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.55), guide = "none") +
  scale_linewidth_manual(values = c(`TRUE` = 1.3, `FALSE` = 0.6), guide = "none") +
  scale_size_manual(values = c(`TRUE` = 3.4, `FALSE` = 2.4), guide = "none") +
  guides(colour = guide_legend(override.aes = list(linewidth = 1.4, size = 0))) +
  labs(
    title = "The pandemic rewrote the leaderboard.",
    subtitle = paste0(
      "Top global causes of disease burden (DALYs in millions), ranked across three years.\n",
      "COVID-19 went from non-existent in 2010 to the world's #1 cause of healthy years lost in 2021.\n",
      "Low back pain and diabetes climbed into the top 10. HIV/AIDS, malaria and diarrheal diseases fell out."),
    caption = paste(
      "Source: IHME, Global Burden of Disease Study 2021 — \"Findings from the GBD 2021 Study\" (May 2024), p. 14.",
      "Original: GBD 2021 DALYs and HALE Collaborators, The Lancet 403:2133, 2024.  Values are absolute DALYs in millions.",
      "#30DayChartChallenge · Day 30 · Global Health Data Exchange Data Day  |  Sharfudeen Yasar Arafath",
      sep = "\n"),
    x = NULL, y = NULL
  ) +
  theme_minimal(base_family = "outfit", base_size = 13) +
  theme(
    plot.background    = element_rect(fill = bg_col, colour = NA),
    panel.background   = element_rect(fill = bg_col, colour = NA),
    panel.grid         = element_blank(),
    axis.text          = element_blank(),
    axis.ticks         = element_blank(),
    legend.position    = "top",
    legend.justification = "left",
    legend.text        = element_text(family = "outfit", size = 10,
                                      colour = text_col),
    legend.key         = element_rect(fill = bg_col, colour = NA),
    legend.background  = element_rect(fill = bg_col, colour = NA),
    legend.margin      = margin(b = 8),
    plot.title         = element_text(family = "outfit", face = "bold",
                                      size = 26, colour = text_col,
                                      margin = margin(b = 6)),
    plot.subtitle      = element_text(family = "roboto_condensed",
                                      size = 12, colour = subtext,
                                      lineheight = 1.3,
                                      margin = margin(b = 4)),
    plot.caption       = element_text(family = "jetbrains", size = 8.5,
                                      colour = subtext, hjust = 0,
                                      lineheight = 1.5,
                                      margin = margin(t = 22)),
    plot.caption.position = "plot",
    plot.title.position   = "plot",
    plot.margin = margin(28, 32, 24, 32)
  )

agg_png("../../chart/day_30_ghdx.png",
        width = 14, height = 11, units = "in", res = 300,
        background = bg_col)
print(p)
invisible(dev.off())
cat("Saved chart/day_30_ghdx.png\n")
